In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import string
import json

import spacy
from spacy.tokens import Doc

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Load Data

In [ ]:
news = pd.read_csv("news_tokenized.csv")

In [ ]:
news["category"].nunique()

41

In [ ]:
news.head()

,category,text
0,ARTS,thing know cindy sherman look untitled film wo...
1,ARTS,trip bountiful foote play look past inspiratio...
2,ARTS,dream care painting capture innocence childhoo...
3,ARTS,w souleo dance exhibition celebrate first look...
4,ARTS,high def selfie arts humanities technology fac...


In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(news["text"])

In [ ]:
print(news.shape)

(37960, 2)


In [ ]:
print(tfidf_matrix.shape)

(37960, 38857)


# Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier
!pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

In [ ]:
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    tfidf_matrix, # sparse matrix
    news["category"], # target
    test_size=0.2,
    stratify=news["category"], # preserve data distribution in both train and test set
    random_state=42)

# Apply SMOTE to balance the dataset: Since some categories have very few samples
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

## train

In [ ]:
def train_logistic_regression(X, y):
    log_reg = OneVsRestClassifier(LogisticRegression(max_iter=500))
    log_reg.fit(X, y)
    return log_reg

In [ ]:
model = train_logistic_regression(X_train_resampled, y_train_resampled)

## evaluation

In [ ]:
from sklearn.metrics import classification_report

# pred
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

                precision    recall  f1-score   support

          ARTS       0.33      0.21      0.25        34
ARTS & CULTURE       0.15      0.15      0.15        53
  BLACK VOICES       0.42      0.49      0.45       167
      BUSINESS       0.38      0.52      0.44       205
       COLLEGE       0.48      0.65      0.55        37
        COMEDY       0.48      0.49      0.49       186
         CRIME       0.45      0.60      0.52       113
CULTURE & ARTS       0.54      0.33      0.41        43
       DIVORCE       0.78      0.65      0.71       137
     EDUCATION       0.28      0.44      0.34        36
 ENTERTAINMENT       0.63      0.61      0.62       591
   ENVIRONMENT       0.33      0.21      0.26        57
         FIFTY       0.11      0.17      0.13        42
  FOOD & DRINK       0.61      0.61      0.61       253
     GOOD NEWS       0.21      0.24      0.22        42
         GREEN       0.36      0.41      0.38        82
HEALTHY LIVING       0.20      0.27      0.23  